# പ്രിഒരിറ്റി അംഗ മിഡിൽവെയർ ഉപയോഗിച്ച് ഹോട്ടൽ ബുക്കിംഗ്

Microsoft Agent Framework ഉപയോഗിച്ച് **ഫംഗ്ഷൻ-ബേസ്ഡ് മിഡിൽവെയർ** ഇതിൽ പ്രദർശിപ്പിക്കുന്നു. നിബന്ധനാത്മക വർക്ഫ്ലോ ഉദാഹരണത്തിൽ ഒരു മിഡിൽവെയർ ലെയർ ചേർത്ത് പ്രിഒരിറ്റി അംഗങ്ങൾക്ക് പ്രത്യേക അനുമതികൾ നൽകുന്നു.

## നിങ്ങൾ പഠിക്കുന്നതെന്ത്:
1. **ഫംഗ്ഷൻ-ബേസ്ഡ് മിഡിൽവെയർ**: ഫംഗ്ഷൻ ഫലങ്ങൾ ഇടപെട്ടു തിരുത്തൽ
2. **കോൺടെക്‌സ്‌റ്റ് ആക്‌സസ്**: നിർവഹണത്തിനുശേഷം `context.result` വായിക്കുകയും തിരുത്തുകയും ചെയ്യുക
3. **ബിസിനസ് ലൊജിക് നടപ്പാക്കൽ**: പ്രിഒരിറ്റി അംഗ લાભങ്ങൾ
4. **ഫലം ഓവർറൈഡ്**: ഉപയോക്തൃ നിലയുടെ അടിസ്ഥാനത്തിൽ ഫംഗ്ഷൻ ഫലങ്ങൾ മാറ്റുക
5. **അതേ വർക്ഫ്ലോ, വ്യത്യസ്ത ഫലങ്ങൾ**: മിഡിൽവെയർ നിയന്ത്രിത പെരുമാറ്റ വ്യത്യാസങ്ങൾ

## മിഡിൽവെയർ ഉപയോ​ഗിച്ചുള്ള വർക്ഫ്ലോ ആർക്കിടെക്ചർ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## നിബന്ധനാത്മക വർക്ഫ്ലോയിൽ നിന്ന് പ്രധാന വ്യത്യാസം:

**മിഡിൽവെയർ ഇല്ലാതെയെങ്കിൽ** (14-conditional-workflow.ipynb):
- പാരിസിൽ മുറികളില്ല → alternative_agent-ലേക്ക് റൂട്ടുചെയ്യുക

**മിഡിൽവെയർ കൂടിയതായി** (ഈ നോട്ട്ബുക്ക്):
- സാധാരണ ഉപയോക്താവ് + പാരിസ് → മുറികളില്ല → alternative_agent-ലേക്ക് റൂട്ടുചെയ്യുക
- പ്രിഒരിറ്റി ഉപയോക്താവ് + പാരിസ് → 🌟 മിഡിൽവെയർ ഓവർറൈഡ്! → ലഭ്യമാണ് → booking_agent-ലേക്ക് റൂട്ടുചെയ്യുക

## മുൻകരുതലുകൾ:
- Microsoft Agent Framework ഇൻസ്റ്റാൾ ചെയ്തു കഴിഞ്ഞിരിക്കണം
- നിബന്ധനാത്മക വർക്ഫ്ലോകളുടെ ബോധ്യം (കാണുക 14-conditional-workflow.ipynb)
- GitHub ടോക്കൺ അല്ലെങ്കിൽ OpenAI API കീ
- മിഡിൽവെയർ മാതൃകകളുടെ അടിസ്ഥാന ജ്ഞാനം


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Step 1: ഘടനയുള്ള പുറത്താക്കലുകൾക്കായി പൈഡാൻറിക് മോഡലുകൾ നിർവചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകുന്ന **സ്കീമ** നിർവചിക്കുന്നു. മിഡിൽവെയർ ലഭ്യത ഫലം മാറ്റുമ്പോൾ ട്രാക്ക് ചെയ്യാൻ `priority_override` ഫീൽഡ് ചേർത്തിട്ടുണ്ട്.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: പ്രാധാന്യമുള്ള അംഗങ്ങളുടെ ഡാറ്റാബേസ് നിർവചിക്കുക

ഈ ഡെമോയ്ക്ക്, നാം ഒരു പ്രാധാന്യമുള്ള അംഗത്വ ഡാറ്റാബേസ് സിമുലേറ്റ് ചെയ്യും. പ്രൊഡക്ഷനിൽ, ഇത് യഥാർത്ഥ ഡാറ്റാബേസ് അല്ലെങ്കിൽ API-യുമായി ക്വറി ചെയ്യുമെന്ന് കരുതാം.

**പ്രാധാന്യമുള്ള അംഗങ്ങൾ:**
- `alice@example.com` - VIP അംഗം
- `bob@example.com` - പ്രീമിയം അംഗം  
- `priority_user` - ടെസ്റ്റ് അക്കൗണ്ട്


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ഘട്ടം 3: ഹോട്ടൽ ബുക്കിംഗ് ഉപകരണം സൃഷ്ടിക്കുക

കണ്ടീഷണൽ വർക്ക്‌ഫ്ലേയുടെ പോലെ തന്നെയാണ്, പക്ഷേ ഇപ്പോൾ ഇത് മിഡിൽവേയർ വഴി തടയപ്പെടും!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 പ്രാധാന്യ പരിശോധന മിഡിൽവെയർ സൃഷ്ടിക്കൽ (പ്രധാന ഫീച്ചർ!)

ഇത് ഈ നോട്ട്ബുക്കിന്റെ **മുൻനിര പ്രവർത്തനക്ഷമത** ആണ്. മിഡിൽവെയർ:

1. **ഇന്റെർസെപ്റ്റ് ചെയ്യുന്നു** hotel_booking ഫംഗ്ഷൻ കോളിനെ
2. `next(context)` വിളിച്ച് ഫംഗ്ഷൻ സാധാരണമാക്കുന്നു
3. `context.result` ൽ ഫലം **പരിശീലിക്കുന്നു**
4. ഉപയോക്താവ് പ്രാധാന്യമുള്ളവര്‍ ആണെങ്കില്‍ കിടക്കയില്ലാതിരുന്നാല്‍ ഫലം **മാറ്റിസ്ഥാപിക്കുന്നു**
5. മാറ്റിയ ഫലം ഏജന്റിന് തിരിച്ചുകാണിക്കുന്നു

**പ്രധാന പാറ്റേൺ:**
```python
async def my_middleware(context, next):
    await next(context)  # ഫംഗ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    # ഇപ്പോൾ context.result ഫംഗ്ഷന്റെ ഫലമാണ് ഉൾക്കൊള്ളുന്നത്
    if some_condition:
        context.result = new_value  # ഒവർറൈഡ്!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Step 5: റൂട്ടിംഗിനായുള്ള നിബന്ധനാ ഫംഗ്ഷനുകൾ നിർദ്ദിഷ്‌ടമാക്കുക

സമാന നിബന്ധനാ ഫംഗ്ഷനുകൾ സാന്റിയൽ വർക്ക്‌ഫ്ലോവിനും - അവ ഘടനാപരമായ ഔട്ട്പുട്ട് പരിശോധിച്ച് റൂട്ടിംഗ് നിർണ്ണയിക്കുന്നു.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Step 6: സ്വന്തം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

മുമ്പത്തെ പോലെ തന്നെ എക്സിക്യൂട്ടർ - അന്തിമ വർക്ക്‌ഫ്ലോ ഔട്ട്പുട്ട് പ്രദർശിപ്പിക്കുന്നു.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Step 7: ലോഡ് പരിസ്ഥിതി വ്യവസ്ഥകൾ

LLM ക്ലയന്റിനെ ക്രമീകരിക്കുക (GitHub മോഡലുകൾ അല്ലെങ്കിൽ OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## പടി 8: മിഡിൽവെയർ ഉപയോഗിച്ച് AI ഏജന്റുകൾ സൃഷ്ടിക്കുക

**പ്രധാന വ്യത്യാസം:** availability_agent സൃഷ്ടിക്കുമ്പോൾ, നാം `middleware` പാരാമീറ്റർ നൽകുന്നു!

ഏജന്റിന്റെ ഫങ്ക്ഷൻ വിളിക്കലിന്റെ പൈപ്പ്‌ലൈൻ keessatti priority_check_middleware എടുത്തു ചേർക്കുന്നത് ഇങ്ങനെ ആണ്.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: Build the Workflow

മുൻപത്തെ പോലെയുള്ള പ്രവൃത്തി പ്രവാഹ ഘടന - ലഭ്യതയുടെ അടിസ്ഥാനത്തിൽ നിബന്ധനാപൂർവമായ റൂട്ടിംഗ്.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ഘട്ടം 10: ടെസ്റ്റ് കേസ് 1 - പാരിസിലെ സാധാരണ ഉപയോക്താവ് (ഓവർറൈഡ് ഇല്ല)

ഒരു സാധാരണ ഉപയോക്താവ് പാരിസ് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → okenn മുറികൾ → alternative_agent-ലേക്ക് റൂട്ടുചെയ്യുന്നു


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 Parisയിലെ പ്രാധാന്യമുള്ള ഉപയോക്താവ് (OVERRIDE ഒപ്പം!)

ഒരു പ്രാഥമിക അംഗം Paris ബുക്കു ചെയ്യാൻ ശ്രമിക്കുന്നു → ആദ്യം മുറികൾ ഇല്ല → 🌟 മിഡിൽവെയർ ഓവർറൈഡ് ചെയ്യുന്നു! → booking_agent ലേക്ക് റൂട്ടുചെയ്യുന്നു

**ഇതാണ് മിഡിൽവെയർ ശക്തിയുടെ പ്രധാന പ്രദർശനം!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Step 12: Test Case 3 - സ്റ്റോക്ക്‌ഹോംലുള്ള പ്രാധാന്യമുള്ള ഉപയോക്താവ് (ഇപ്പോൾ ലഭ്യമാണ്)

പ്രാധാന്യമുള്ള ഉപയോക്താവ് സ്റ്റോക്ക്‌ഹോം ശ്രമിക്കുന്നു → മുറികൾ ലഭ്യമാണ് → ഒതുക്കം ആവശ്യം ഇല്ല → booking_agentലേക്ക് റൂട്ടുകൾ

ഇത് കാണിക്കുന്നത് മിഡിൽവെയർ ആവശ്യത്തിന് മാത്രമേ പ്രവർത്തിക്കൂ എന്നതാണ്!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## പ്രധാന കണ്ടെത്തലുകളും മിഡിൽവെയർ ആശയങ്ങളും

### ✅ നിങ്ങൾ പഠിച്ചു:

#### **1. ഫംഗ്ഷൻ-അടിസ്ഥാന മിഡിൽവെയർ പാറ്റേൺ**

മിഡിൽവെയർ ഒരു ലളിതമായ അസിങ്ക്രൺ ഫംഗ്ഷൻ ഉപയോഗിച്ച് ഫംഗ്ഷൻ കോൾസ് ചെന്ന് തടയുന്നു:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ഫംഗ്ഷൻ പ്രവർത്തനം ആരംഭിക്കുന്നത് മുൻപ്
    print("Intercepting...")
    
    # ഫംഗ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    await next(context)
    
    # ഫംഗ്ഷൻ പ്രവർത്തനം കഴിഞ്ഞ് - ഫലം പരിശോധിക്കുക
    if context.result:
        # ആവശ്യമെങ്കിൽ ഫലം മാറ്റം നടത്തുക
        context.result = modified_value
```

#### **2. കോൺടെക്സ്റ്റ് ആക്‌സസ് ചെയ്യൽ ಮತ್ತು ഫലം ഒവർറൈഡ് ചെയ്യൽ**

- `context.function` - കോൾ ചെയ്യപ്പെടുന്ന ഫംഗ്ഷൻ ആക്‌സസ് ചെയ്യുക
- `context.arguments` - ഫംഗ്ഷൻ ആർത്തങ്ങളൂം വായിക്കുക
- `context.kwargs` - അധിക പാരാമീറ്ററുകൾ ആക്‌സസ് ചെയ്യുക
- `await next(context)` - ഫംഗ്ഷൻ നിർവഹിക്കുക
- `context.result` - ഫംഗ്ഷന്റെ ഔട്ട്‌പുട്ട് വായിക്കുകയോ/മാറ്റുകയോ ചെയ്യുക

#### **3. ബിസിനസ്സ് ലൊജിക് നടപ്പിലാക്കൽ**

നമ്മുടെ മിഡിൽവെയർ പ്രഥമത അംഗബാധ്യതകൾ നടപ്പിലാക്കുന്നു:
- **സാധാരണ ഉപയോക്താക്കൾ**: മാറ്റങ്ങൾ ഇല്ല, സാധാരണ പ്രവൃത്തി പ്രവাহം
- **പ്രാഥമിക ഉപയോക്താക്കൾ**: "ലഭ്യത ഇല്ല" → "ലഭ്യമാണെന്ന്" ഒവർറൈഡ് ചെയ്യൽ
- **ശരിയായപ്പോൾ മാത്രം ഒവർറൈഡ് ചെയ്യൽ**: വ്യവസ്ഥിത ലൊജിക്

#### **4. ഒരേ പ്രവൃത്തി പ്രവാഹം, വ്യത്യസ്ത ഫലങ്ങൾ**

മിഡിൽവെയറിന്റെ ശക്തി:
- ✅ പ്രവൃത്തി പ്രവാഹത്തിൽ മാറ്റങ്ങൾ ഇല്ല
- ✅ ടൂൾ ഫംഗ്ഷനിൽ മാറ്റങ്ങൾ ഇല്ല
- ✅ വ്യവസ്ഥിത റൂട്ടിംഗ് ലൊജികിൽ മാറ്റങ്ങൾ ഇല്ല
- ✅ മിഡിൽവെയർ മാത്രം → വ്യത്യസ്ത പെരുമാറ്റം!

### 🚀 യാഥാർത്ഥ്യ ലോക സംവിധാനങ്ങൾ:

1. **വിഐപി/പ്രീമിയം ഫീച്ചറുകൾ**
   - പ്രീമിയം ഉപയോക്താക്കൾക്കുള്ള നിരക്ക് പരിധി ഒവർറൈഡ് ചെയ്യൽ
   - പ്രയോഗങ്ങൾക്ക് പ്രാഥമിക ആക്‌സസ് നൽകൽ
   - പ്രീമിയം ഫീച്ചറുകൾ ഡൈനാമിക് ആയി തുറക്കൽ

2. **എ/ബി ടെസ്റ്റിംഗ്**
   - ഉപയോക്താക്കളെ വ്യത്യസ്ത നടപ്പിലാക്കലുകളിലേക്ക് റൂട്ടുചെയ്യുക
   - പുതിയ ഫീച്ചറുകൾ പ്രത്യേക ഉപയോക്താക്കളിൽ പരിശോധന നടത്തുക
   - മുന്നത്തരം ഫീച്ചർ റിലീസുകൾ

3. **സുരക്ഷ & അനുസരണ മാനദണ്ഡങ്ങൾ**
   - ഫംഗ്ഷൻ കോൾസ് ഓഡിറ്റ് ചെയ്യുക
   - ലളിതവും സുരക്ഷിതവും പ്രവർത്തനങ്ങൾ തടയുക
   - ബിസിനസ്സ് ചട്ടങ്ങൾ കർശനമായി പാലിക്കുക

4. **പ്രവർത്തനക്ഷമത മെച്ചപ്പെടുത്തൽ**
   - പ്രത്യേക ഉപയോക്താക്കൾക്ക് ഫലം ക്യാഷ് ചെയ്യുക
   - സാധ്യമായപ്പോൾ ചെലവേറിയ പ്രവർത്തനങ്ങൾ ഒഴിവാക്കുക
   - ഡൈനാമിക് റിസോഴ്‌സ് വിനിയോഗം

5. **പിശക് കൈകാര്യം & റിട്രി**
   - പിശകുകൾ പിടിച്ചു മനസിലാക്കുക
   - റിട്രി ലൊജിക് നടപ്പിലാക്കുക
   - വ്യത്യസ്ത നടപ്പിലാക്കലിലേക്ക് ഫാൾബാക്ക്

6. **ലോഗിംഗ് & നിരീക്ഷണം**
   - ഫംഗ്ഷൻ നിർവഹണ സമയങ്ങൾ ട്രാക്ക് ചെയ്യുക
   - പാരാമീറ്ററുകളും ഫലങ്ങളും ലോഗ് ചെയ്യുക
   - ഉപയോഗ പതിപ്പുകൾ നിരീക്ഷിക്കുക

### 🔑 ഡെക്കറേറ്ററുകളിൽ നിന്ന് പ്രധാന വ്യത്യാസങ്ങൾ:

| ഫീച്ചർ | ഡെക്കറേറ്റർ | മിഡിൽവെയർ |
|---------|-------------|------------|
| **സ്‌കോപ്പ്** | ഒറ്റ ഫംഗ്ഷൻ | ഏജന്റ് ഉള്ളിലെ എല്ലാ ഫംഗ്ഷനുകളും |
| **മറുപടി വൈവിധ്യം** | നിർവചിക്കപ്പെട്ടപ്പോൾ സ്ഥിരം | റണ്‍ടൈമിൽ ഡൈനാമിക് |
| **കോൺടെക്സ്റ്റ്** | പരിമിതം | പൂർണ്ണ ഏജന്റ് കോൺടെക്സ്റ്റ് |
| **സംയോജനം** | പല ഡെക്കറേറ്ററുകൾ | മിഡിൽവെയർ പൈപ്പ്‌ലൈൻ |
| **ഏജന്റ്-ആവെയർ** | ഇല്ല | ഉണ്ട് (ഏജന്റ് സ്റ്റേറ്റിലേക്ക് ആക്‌സസ്) |

### 📚 മിഡിൽവെയർ എപ്പോൾ ഉപയോഗിക്കണം:

✅ **ഉപയോഗിക്കുക:**
- ഉപയോക്താവ്/സെഷൻ അവസ്ഥ അടിസ്ഥാനമാക്കിയുള്ള പെരുമാറ്റം മാറ്റേണ്ടത്
- പല ഫംഗ്ഷനുകളിലും ലോഗിക് പ്രയോഗിക്കേണ്ടത്
- ഏജന്റ് തലത്തിലുള്ള കോൺടെക്സ്റ്റിൻറെ ആക്‌സസ് വേണമെന്ന്
- ക്രാസ്-കട്ടിംഗ് കാരണങ്ങൾ (ലോഗിംഗ്, ഓതൻ്റിക്കേഷൻ തുടങ്ങിയവ) നടപ്പിലാക്കുന്നത്

❌ **ഉപയോഗിക്കരുത്:**
- ലളിതമായ ഇൻപുട്ട് പരിശോധന (പൈഡാന്റിക് ഉപയോഗിക്കുക)
- ഫംഗ്ഷൻ സ്പെസിഫിക് ലൊജിക് (ഫംഗ്ഷനിൽ തന്നെ ഉണ്ടാക്കുക)
- ഒറ്റ തവണ മാറ്റങ്ങൾ (സരളമായി ഫംഗ്ഷൻ മാറ്റുക)

### 🎓 മെച്ചപ്പെട്ട പാറ്റേണുകൾ:

```python
# പല മിഡിൽവെയറുകളും (പ്രവർത്തനക്രമം പ്രധാനമാണ്!)
middleware=[
    logging_middleware,      # ആദ്യമേ ലോഗുകൾ
    auth_middleware,         # ശേഷം ഓത്ത് പരിശോധിക്കുന്നു
    cache_middleware,        # പിന്നീട് ക്യാഷേ പരിശോധിക്കുന്നു
    rate_limit_middleware,   # തുടർന്ന് നിരക്ക് പരിമിതി
    priority_check_middleware  # ഒടുവിൽ മുൻഗണന പരിശോധിക്കൽ
]

# നിബന്ധനാ അടിസ്ഥാനത്തിലുള്ള മിഡിൽവെയർ പ്രവർത്തനം
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ഫലം മാറ്റം
    else:
        # പ്രവർത്തനം പൂർണ്ണമായി ഒഴിവാക്കുക
        context.result = cached_value
```

### 🔗 ബന്ധപ്പെട്ട ആശയങ്ങൾ:

- **ഏജന്റ് മിഡിൽവെയർ**: agent.run() കോൾസ് തടയുക
- **ഫംഗ്ഷൻ മിഡിൽവെയർ**: ടൂൾ ഫംഗ്ഷൻ കോൾസ് തടയുക (നാം ഉപയോഗിച്ചത്!)
- **മിഡിൽവെയർ പൈപ്പ്‌ലൈൻ**: ക്രമത്തിൽ പ്രവര്ത്തിക്കുന്ന മിഡിൽവെയറുകളുടെ ശ്രേണി
- **കോൺടെക്സ്റ്റ് പ്രൊപ്പഗേഷൻ**: മിഡിൽവെയർ ശ്രേണി വഴി സ്റ്റേറ്റ് പാസ്സുചെയ്യുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
